# Import + Turn into geojson

In [1]:
import pandas as pd

In [2]:
dfsight = pd.read_csv('../../data/v6/HBW_VF_LL.csv')
# ("../../data/v5_FE/all_species_expanded_raw.csv")

                   
dfsight['sighting_date_and_time_utc'] = pd.to_datetime(dfsight['sighting_date_and_time_utc'], errors='coerce', format='mixed', utc=True)


In [3]:
dfsight.head()

,creator,species_code,sighting_date_and_time_utc,track_key,system_filename,variable_data,latitude,longitude,activity_codes,boat_name,...,animals_adult,animals_juvenile,animals_calf,animals_unknown_age,animals_total,inter_species_interaction,general_notes,sight_distance,official_departure_time,recorder_name
0,9879626d-84f0-441d-adfa-0ce677417663,HBW,2011-03-14 18:30:00+00:00,NaN,NaN,"{""activity_codes"":[],""boat_name"":""Ocean Intrig...",20.754083,-156.503967,[],Ocean Intrigue,...,2.0,NaN,1.0,NaN,3,NaN,"1.5 miles out of harbor.\r\nMom, calf, escort ...",NaN,NaN,Jessica Rickard
1,9879626d-84f0-441d-adfa-0ce677417663,HBW,2011-11-20 00:23:00+00:00,NaN,NaN,"{""activity_codes"":[""MS""],""boat_name"":""Ocean Vo...",20.765083,-156.614550,"[""MS""]",Ocean Voyager,...,1.0,NaN,NaN,NaN,1,NaN,slowly cruising whale towards lahaina,NaN,NaN,Hollis Turner
2,9879626d-84f0-441d-adfa-0ce677417663,HBW,2011-11-20 00:35:00+00:00,NaN,NaN,"{""activity_codes"":[""FS""],""boat_name"":""Ocean Vo...",20.770133,-156.572267,"[""FS""]",Ocean Voyager,...,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,Hollis Turner
3,9879626d-84f0-441d-adfa-0ce677417663,HBW,2011-11-23 18:30:00+00:00,NaN,NaN,"{""activity_codes"":[""MA""],""boat_name"":""Ocean Di...",20.857717,-156.734967,"[""MA""]",Ocean Discovery,...,NaN,1.0,NaN,NaN,1,NaN,Mid-channel between Lahaina and Lana'i. Sub ad...,NaN,NaN,Jon Ciser
4,9879626d-84f0-441d-adfa-0ce677417663,HBW,2011-11-23 23:24:00+00:00,NaN,NaN,"{""activity_codes"":[""MR""],""boat_name"":""Ocean Od...",20.745283,-156.661300,"[""MR""]",Ocean Odyssey,...,1.0,NaN,NaN,NaN,1,NaN,We first observed this humpback whale about 5 ...,NaN,NaN,Travis Farrell


# Handle Malformed Lat,Long

In [4]:
# Import + Turn into GeoJSON
import pandas as pd
import re
import numpy as np

def parse_coord(coord):
    if pd.isna(coord):
        return np.nan

    s = str(coord).strip()

    # Junk values
    if s in ('.', '') or 'note' in s.lower():
        return np.nan

    # Explicit zero
    try:
        if float(s) == 0.0:
            return 0.0
    except ValueError:
        pass

    # Handle malformed coordinates such as:
    # "-156.35.245" -> -156 degrees, 35.245 minutes
    # "156 30.238"  -> 156 degrees, 30.238 minutes
    #
    # IMPORTANT:
    # Dontforce longitude negative because this dataset
    # contains coordinates from global-disiimilar from the prepsightings data tab
    match = re.match(r'^(-?)(\d+)[ .](\d+\.\d+)$', s)

    if match:
        sign, deg, minutes = match.groups()

        decimal = float(deg) + float(minutes) / 60

        if sign == '-':
            decimal = -decimal

        return decimal

    # Already a clean numeric coordinate
    try:
        return float(s)

    except ValueError:
        return np.nan


# ---------------------------------------------------------
# Keep original coordinates for inspection
# ---------------------------------------------------------

dfsight['longitude_raw'] = dfsight['longitude']
dfsight['latitude_raw'] = dfsight['latitude']


# ---------------------------------------------------------
# Convert coordinates
# ---------------------------------------------------------

dfsight['longitude'] = dfsight['longitude'].apply(parse_coord)
dfsight['latitude'] = dfsight['latitude'].apply(parse_coord)


# ---------------------------------------------------------
# Validate coordinate ranges
# ---------------------------------------------------------

invalid_coords = (
    dfsight['longitude'].isna() |
    dfsight['latitude'].isna() |
    (dfsight['longitude'] < -180) |
    (dfsight['longitude'] > 180) |
    (dfsight['latitude'] < -90) |
    (dfsight['latitude'] > 90)
)

print(f"Original rows: {len(dfsight):,}")
print(f"Invalid coordinate rows: {invalid_coords.sum():,}")


# ---------------------------------------------------------
# Save invalid rows for inspection
# ---------------------------------------------------------

dropped_coords = dfsight.loc[
    invalid_coords,
    [
        'latitude_raw',
        'longitude_raw'
    ] + (
        ['sighting_id']
        if 'sighting_id' in dfsight.columns
        else []
    )
].copy()

dropped_coords = dropped_coords.rename(
    columns={
        'latitude_raw': 'latitude',
        'longitude_raw': 'longitude'
    }
)


# ---------------------------------------------------------
# Remove invalid coordinates
# ---------------------------------------------------------

dfsight = dfsight.loc[~invalid_coords].copy()

dfsight = dfsight.drop(
    columns=['latitude_raw', 'longitude_raw']
)


print(f"Rows removed: {len(dropped_coords):,}")
print(f"Clean rows: {len(dfsight):,}")


Original rows: 34,085
Invalid coordinate rows: 0
Rows removed: 0
Clean rows: 34,085


# Clean animals total data

parse_coord —  general malformed-coordinate parser (kept hemisphere-agnostic, since  noted this dataset isn't Hawaii-only, unlike the DMM version tied to Maui Nui).
clean_coordinates — applies parse_coord to both columns, validates ranges, splits into (clean_df, dropped_df).
coerce_animal_counts — forces the animal columns numeric, fills NaN with 0.
drop_animal_outliers — drops rows with any animal count outside a min/max range, returns what got dropped.
reconcile_animal_totals — fixes mismatches between animals_total and the sum of its parts (bumps total up, or assigns the difference to unknown_age), same logic as the notebook's mismatch block.
drop_zero_sightings — drops rows with animals_total <= 0.
to_geojson — the GeoDataFrame export had commented out.

load_bathymetry_grid / lookup_depths — same GEBCO loading and vectorized nearest-neighbor lookup from the original notebook.
exclude_land_points — the new piece: drops any row whose (lat, lon) resolves to positive (land) elevation, treating (0, 0) as the "no sighting" sentinel so it's never mistakenly looked up or flagged as land. Returns (water_df, land_df) so you can inspect what got dropped.

In [10]:
"""
Reusable cleaning functions for sightings data.

These are the pieces from the original notebook that are NOT tied to a
specific region (Maui Nui, bathymetry, GEBCO, vessel tracks, etc.) —
just lat/long parsing + validation, and animal-count cleaning/reconciliation.

Import and chain them:

    df = pd.read_csv("../../data/v5_FE/all_species_expanded_raw.csv")
    df, dropped = clean_coordinates(df)
    df = coerce_animal_counts(df)
    df, outliers = drop_animal_outliers(df)
    df = reconcile_animal_totals(df)
"""

import re
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Coordinate parsing / cleaning
# ---------------------------------------------------------------------------

def parse_coord(coord):
    """
    Parse a single lat or long value that may be malformed.

    Handles:
      - NaN / empty / '.' / anything containing 'note' -> NaN
      - explicit '0' / 0.0 -> kept as exactly 0.0 (sentinel "no sighting")
      - malformed degree.minute strings like "-156.35.245" or "156 30.238"
        -> degrees + minutes/60 (sign preserved from the leading '-' only;
        does NOT force a hemisphere, unlike the region-specific version
        that assumed everything was West Hawaii)
      - already-clean numeric strings -> float(), unchanged sign

    Returns np.nan for anything unparseable.
    """
    if pd.isna(coord):
        return np.nan

    s = str(coord).strip()

    # Junk values
    if s in ('.', '') or 'note' in s.lower():
        return np.nan

    # Explicit zero sentinel
    try:
        if float(s) == 0.0:
            return 0.0
    except ValueError:
        pass

    # Malformed degree.minute pattern: "-156.35.245" or "156 30.238"
    match = re.match(r'^(-?)(\d+)[ .](\d+\.\d+)$', s)
    if match:
        sign, deg, minutes = match.groups()
        decimal = float(deg) + float(minutes) / 60
        if sign == '-':
            decimal = -decimal
        return decimal

    # Already-clean numeric
    try:
        return float(s)
    except ValueError:
        return np.nan


def clean_coordinates(df, lat_col='latitude', lon_col='longitude', id_col='sighting_id'):
    """
    Parse + validate lat/long columns on df.

    Steps:
      1. Keep raw copies of lat/long for anything that gets dropped.
      2. Apply parse_coord to both columns.
      3. Flag rows as invalid if either coord is NaN or out of
         valid range (-90..90 for lat, -180..180 for lon).
      4. Split into (clean_df, dropped_df).

    Returns
    -------
    (clean_df, dropped_df)
        clean_df  : original df, invalid rows removed, lat/long parsed
        dropped_df: the raw (unparsed) lat/long for every dropped row
                    (+ id_col if present in df)
    """
    df = df.copy()
    df['latitude_raw'] = df[lat_col]
    df['longitude_raw'] = df[lon_col]

    df[lat_col] = df[lat_col].apply(parse_coord)
    df[lon_col] = df[lon_col].apply(parse_coord)

    invalid = (
        df[lat_col].isna() | df[lon_col].isna() |
        (df[lat_col] < -90) | (df[lat_col] > 90) |
        (df[lon_col] < -180) | (df[lon_col] > 180)
    )

    keep_cols = ['latitude_raw', 'longitude_raw'] + (
        [id_col] if id_col in df.columns else []
    )
    dropped_df = df.loc[invalid, keep_cols].rename(
        columns={'latitude_raw': lat_col, 'longitude_raw': lon_col}
    ).copy()

    clean_df = df.loc[~invalid].drop(columns=['latitude_raw', 'longitude_raw']).copy()

    print(f"Original rows: {len(df):,}")
    print(f"Invalid coordinate rows: {invalid.sum():,}")
    print(f"Clean rows: {len(clean_df):,}")

    return clean_df, dropped_df


# ---------------------------------------------------------------------------
# Animal-count cleaning
# ---------------------------------------------------------------------------

ANIMAL_COLS = [
    'animals_adult',
    'animals_juvenile',
    'animals_calf',
    'animals_unknown_age',
    'animals_total',
]


def coerce_animal_counts(df, cols=ANIMAL_COLS):
    """
    Force animal-count columns to numeric, filling anything unparseable
    (or missing) with 0.
    """
    df = df.copy()
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    df[cols] = df[cols].fillna(0)
    return df


def get_high_animal_counts(df, col='animals_total', threshold=100):
    """
    Return a separate df of rows where `col` > threshold, for
    inspection only -- doesn't touch or drop anything from `df`.

    Use this before calling drop_animal_outliers() to eyeball what
    you'd actually be dropping and pick a sensible max_val.
    """
    high = df.loc[df[col] > threshold].copy()
    print(f"{len(high):,} rows with {col} > {threshold}")
    return high


def drop_animal_outliers(df, cols=ANIMAL_COLS, max_val=30, min_val=0):
    """
    Drop rows where any animal-count column is outside [min_val, max_val].

    Returns (clean_df, outliers_df) so you can inspect what was removed.
    """
    outliers = df[
        (df[cols] > max_val).any(axis=1) | (df[cols] < min_val).any(axis=1)
    ]
    clean_df = df.drop(outliers.index)
    print(f"Dropped {len(outliers):,} outlier rows (outside [{min_val}, {max_val}])")
    return clean_df, outliers


def reconcile_animal_totals(
    df,
    adult_col='animals_adult',
    juvenile_col='animals_juvenile',
    calf_col='animals_calf',
    unknown_col='animals_unknown_age',
    total_col='animals_total',
):
    """
    Reconcile animals_total against the sum of its parts.

    - If total < sum(parts): bump total up to match the sum.
    - If total > sum(parts): assign the difference to animals_unknown_age.

    Rows that already match are left untouched. Returns a new df with the
    corrected columns.
    """
    df = df.copy()
    parts_sum = df[adult_col] + df[juvenile_col] + df[calf_col] + df[unknown_col]
    mismatch_mask = parts_sum != df[total_col]

    mismatch = df.loc[mismatch_mask].copy()
    sums = mismatch[adult_col] + mismatch[juvenile_col] + mismatch[calf_col] + mismatch[unknown_col]

    # Case 1: total is less than sum of parts -> bump total up to match
    under_total = mismatch[total_col] < sums
    mismatch.loc[under_total, total_col] = sums[under_total]

    # Case 2: total is greater than sum of parts -> assign the difference to unknown_age
    sums_without_unknown = mismatch[adult_col] + mismatch[juvenile_col] + mismatch[calf_col]
    over_total = ~under_total
    miss = mismatch[total_col] - sums_without_unknown
    mismatch.loc[over_total, unknown_col] += miss[over_total]

    df.loc[mismatch.index, total_col] = mismatch[total_col]
    df.loc[mismatch.index, unknown_col] = mismatch[unknown_col]

    print(f"Reconciled {mismatch_mask.sum():,} mismatched rows")
    return df


def drop_zero_sightings(df, total_col='animals_total'):
    """Drop (or just flag) rows with no animals recorded at all."""
    zero_mask = df[total_col] <= 0
    print(f"{zero_mask.sum():,} rows with {total_col} <= 0")
    return df.loc[~zero_mask].copy()


# ---------------------------------------------------------------------------
# GeoJSON export
# ---------------------------------------------------------------------------

def to_geojson(df, out_path, lat_col='latitude', lon_col='longitude'):
    """
    Write df to a GeoJSON file using lat/long as point geometry.
    Requires geopandas.
    """
    import geopandas as gpd

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs="EPSG:4326",
    )
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"Wrote {len(gdf):,} points to {out_path}")
    return gdf


if __name__ == "__main__":
    # Example pipeline matching your script's intent
    df = pd.read_csv("../../data/v5_FE/all_species_expanded_raw.csv")
    df['sighting_date_and_time_utc'] = pd.to_datetime(
        df['sighting_date_and_time_utc'], errors='coerce', format='mixed', utc=True
    )

    df, dropped_coords = clean_coordinates(df)

    df = coerce_animal_counts(df)

    high_counts = get_high_animal_counts(df, threshold=200)  # inspect before dropping
    print(high_counts.head())

    df, animal_outliers = drop_animal_outliers(df, max_val=200)
    df = reconcile_animal_totals(df)
    df = drop_zero_sightings(df)

    print(df.shape)
    print(df.head())
    # If you want GeoJSON later, to_geojson(df, "points_allspeciesv5.geojson")
    # is still available below -- just not called here.

C:\Users\research\AppData\Local\Temp\ipykernel_25576\1599265000.py:242: DtypeWarning: Columns (0: latitude, 1: longitude, 2: match_is_certain, 3: match_data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/v5_FE/all_species_expanded_raw.csv")


Original rows: 46,143
Invalid coordinate rows: 172
Clean rows: 45,971
317 rows with animals_total > 200
      sighting_id                               creator species_code  \
11          47512  14c53063-a2b7-41aa-95e8-e2b20579bdfd          HBW   
683         45781  3646ff33-c102-46a1-8ce0-26f40734aac9          SPN   
1327         1291  9879626d-84f0-441d-adfa-0ce677417663          SPN   
1333         1297  9879626d-84f0-441d-adfa-0ce677417663          SPN   
1409         1371  9879626d-84f0-441d-adfa-0ce677417663          SPT   

       latitude   longitude       sighting_date_and_time_utc  \
11    20.791716 -156.513439 2024-05-21 01:10:53.209000+00:00   
683   20.946487 -156.695308        2023-07-29 23:54:32+00:00   
1327  20.633783  156.490333        2012-05-16 20:50:00+00:00   
1333  20.736350  156.883083        2012-05-21 23:37:00+00:00   
1409  20.781833  156.813667        2012-07-12 20:14:00+00:00   

                               user_sighting_key  schema_version track_key  \


# Save to geojson

In [ ]:
import geopandas as gpd
dfsight=df.copy()
gdf = gpd.GeoDataFrame(
    dfsight,
    geometry=gpd.points_from_xy(
        dfsight['longitude'],
        dfsight['latitude']
    ),
    crs="EPSG:4326"
)


gdf.to_file(
    "points_allspeciesv5.geojson",
    driver="GeoJSON"
)

In [14]:
dfcheck=dfsight.sort_values('animals_total', ascending=False)
dfcheck.head()

,sighting_id,creator,species_code,latitude,longitude,sighting_date_and_time_utc,user_sighting_key,schema_version,track_key,needs_attention,...,animals_adult,animals_juvenile,animals_calf,animals_unknown_age,animals_total,inter_species_interaction,general_notes,sight_distance,official_departure_time,recorder_name
7421,7644,9879626d-84f0-441d-adfa-0ce677417663,SPN,20.780567,156.816333,2014-06-23 19:15:00+00:00,13-17669,1,NaN,NaN,...,120.0,150.0,0.0,0.0,270,NaN,NaN,NaN,0700,jenny heard
44164,47720,cc69fda0-e00e-4f3a-8a49-16a49a72854f,BND,-0.876068,-89.664383,2024-08-10 18:05:52+00:00,32343-64900,1,CetaceaGalapagos_20240810070233,NaN,...,200.0,50.0,10.0,0.0,260,NaN,NaN,NaN,0700,Juan Pablo Muñoz
37715,41187,9d234666-ee1d-4567-b0cf-9cef6d00b8b4,SPN,20.877899,-156.727979,2022-01-25 22:44:26+00:00,173-58369,1,NaN,NaN,...,100.0,0.0,50.0,100.0,250,False,NaN,GT_300,1130,Lexie Jeffers
32716,36152,3646ff33-c102-46a1-8ce0-26f40734aac9,SPN,20.921280,-156.699331,2020-12-03 23:02:39+00:00,157-53333,1,NaN,NaN,...,100.0,0.0,0.0,150.0,250,False,NaN,GT_300,1230,Dan Kraver
20496,23396,3646ff33-c102-46a1-8ce0-26f40734aac9,SPN,20.952000,-156.694000,2017-07-29 17:35:00+00:00,157-34215,1,NaN,NaN,...,100.0,0.0,6.0,106.0,212,NaN,NaN,80_300,0700,Mikhala Reedy


In [15]:
#highest month year 
src = gdf   # the frame you exported to points_allspeciesv5.geojson

d = src['sighting_date_and_time_utc']
season = d.dt.year.where(d.dt.month == 12, d.dt.year - 1)
valid = d.notna() & ~season.isin([2011, 2019, 2026])

dfcheck = (
    src.loc[valid]
       .groupby([d[valid].dt.year.rename('year'), d[valid].dt.month.rename('month')])
       .agg(n_sightings=('sighting_date_and_time_utc', 'size'),
            animals_total=('animals_total', 'sum'))
       .reset_index()
       .assign(month_year=lambda x: pd.to_datetime(dict(year=x.year, month=x.month, day=1))
                                      .dt.strftime('%B %Y'))
       .sort_values('n_sightings', ascending=False)
       .reset_index(drop=True)
)
dfcheck.head(10)

,year,month,n_sightings,animals_total,month_year
0,2017,1,1241,5984,January 2017
1,2018,2,1186,3596,February 2018
2,2017,2,1144,3624,February 2017
3,2018,1,1126,4364,January 2018
4,2022,1,1049,3715,January 2022
5,2016,2,955,3462,February 2016
6,2019,3,931,3941,March 2019
7,2017,3,929,4460,March 2017
8,2014,1,894,6841,January 2014
9,2016,3,893,3998,March 2016
